# RoBiS — MVTec AD 2 — Single-Class — ONE CATEGORY / RUN

Notebook này implement lại **đầy đủ pipeline RoBiS** theo repo chính thức `xrli-U/RoBiS`, nhưng tổ chức theo đúng kiểu notebook bạn gửi: **mỗi runtime chỉ train / infer 1 category**, checkpoint và metric lưu trên Google Drive để tích lũy dần 8 category.

Pipeline:

```text
MVTec AD 2 native
    ↓
Swin-Cropping 1024, overlap 0.1
    ↓
INP-Former single-class
DINOv2-R ViT-B/14
+ exposure/noise augmentation
    ↓
continuous anomaly maps 256×256 / crop
    ↓
merge về 1/4 native resolution
    ↓
AU-PRO@0.05 trên test_public
    ↓
MEBin ∪ (mean + 3σ)
    ↓
SAM-Finer
    ↓
SegF1 / ClassF1 trên test_public
    ↓
(optional) test_private + test_private_mixed
    ↓
submission folder / tar.gz
```

**Điểm giữ nguyên từ repo gốc**
- Encoder: `dinov2reg_vit_base_14`
- Input/Crop: `518 × 518`
- `INP_num = 6`
- 200 epochs, AdamW-compatible optimizer `StableAdamW`
- LR `1e-3 → 1e-4`, warmup 100 iterations
- Swin-Cropping: window `1024`, overlap `0.1`
- Output crop map: `256 × 256`
- Gaussian smoothing: kernel `5`, sigma `4`
- Binarization: `MEBin OR mean+3std`
- SAM-H: `test_public`, `test_private`
- SAM-B: `test_private_mixed`
- `rice`: **không dùng SAM-Finer**, đúng code gốc

**Khác biệt chỉ để chạy ổn định trên Colab**
- Không ép Torch 2.0.1/Python 3.8 vì Colab hiện tại có thể không tương thích wheel cũ.
- Giữ Torch/CUDA sẵn có của Colab, chỉ pin các package phụ.
- Thêm checkpoint resume mỗi epoch.
- Inference được viết dạng streaming để tránh giữ toàn bộ anomaly maps trên GPU; phép chuẩn hóa global 0–255 vẫn giữ đúng logic repo.


## 1. Mount Drive + chọn **1 category**

Cấu trúc Drive được giữ giống notebook INP-Former bạn gửi:

```text
MyDrive/
└── [Q3-4] 2026/
    └── [S7] Computer Vision/
        └── CV-Nhóm 9/
            └── data/
                ├── can.tar.gz
                ├── fabric.tar.gz
                ├── fruit_jelly.tar.gz
                ├── rice.tar.gz
                ├── sheet_metal.tar.gz
                ├── vial.tar.gz
                ├── wallplugs.tar.gz
                └── walnuts.tar.gz
```

Mỗi lần chỉ đổi biến `CATEGORY`, rồi **Restart runtime + Run all**.


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path, PurePosixPath
import os, sys, json, shutil, subprocess, hashlib, tarfile, random

PROJECT_DRIVE = Path(
    "/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9"
)

ARCHIVE_DIR = PROJECT_DRIVE / "data"
DRIVE_OUT = PROJECT_DRIVE / "robis_results_single_class"
WEIGHT_CACHE = PROJECT_DRIVE / "robis_weight_cache"

# ============================================================
# CHỈ SỬA CATEGORY
# ============================================================

CATEGORY = "can"
# CATEGORY = "fabric"
# CATEGORY = "fruit_jelly"
# CATEGORY = "rice"
# CATEGORY = "sheet_metal"
# CATEGORY = "vial"
# CATEGORY = "wallplugs"
# CATEGORY = "walnuts"

ALL_CATEGORIES = [
    "can",
    "fabric",
    "fruit_jelly",
    "rice",
    "sheet_metal",
    "vial",
    "wallplugs",
    "walnuts",
]

assert CATEGORY in ALL_CATEGORIES

# ============================================================
# OFFICIAL RoBiS SETTINGS
# ============================================================

ROBIS_COMMIT = "93b031698df828f13ab79bfb6c5bf30b8e4d6e65"

INPUT_SIZE = 518
CROP_SIZE = 518
INP_NUM = 6

TRAIN_BATCH = 16
EPOCHS = 200

INFER_BATCH = 8
SAFE_NUM_WORKERS = 4
SAVE_EVERY = 10

SEED = 1

SWIN_WINDOW = 1024
SWIN_OVERLAP = 0.1

MAP_SIZE = 256

# ============================================================
# FLAGS
# ============================================================

FORCE_EXTRACT = False
FORCE_CROP = False
FORCE_TRAIN = False
FORCE_PUBLIC_INFERENCE = False

RUN_PUBLIC_EVAL = True
RUN_MEBIN = True
RUN_SAM_FINER = True

# Chỉ bật khi muốn tạo output server cho private/private_mixed.
RUN_PRIVATE_INFERENCE = False

# ============================================================
# LOCAL PATHS
# ============================================================

REPO = Path("/content/RoBiS")
MAD2_NATIVE_ROOT = REPO / "data" / "mvtec_ad_2"
PROCESSED_ROOT = REPO / "mvtec_ad_2_processed"

LOCAL_RUN = REPO / "_run"
LOCAL_AMAP_ROOT = LOCAL_RUN / "anomaly_map_results"

# ============================================================
# DRIVE OUTPUTS
# ============================================================

DRIVE_OUT.mkdir(parents=True, exist_ok=True)
WEIGHT_CACHE.mkdir(parents=True, exist_ok=True)

CATEGORY_OUT = DRIVE_OUT / CATEGORY
CATEGORY_OUT.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("CATEGORY       :", CATEGORY)
print("ARCHIVE_DIR    :", ARCHIVE_DIR)
print("DRIVE_OUT      :", DRIVE_OUT)
print("ROBIS_COMMIT   :", ROBIS_COMMIT)
print("=" * 72)


Mounted at /content/drive
CATEGORY       : can
ARCHIVE_DIR    : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/data
DRIVE_OUT      : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/robis_results_single_class
ROBIS_COMMIT   : 93b031698df828f13ab79bfb6c5bf30b8e4d6e65


## 2. Clone đúng RoBiS commit + môi trường Colab

Repo gốc khai báo Python 3.8 / CUDA 11.7 / Torch 2.0.1. Notebook này **không thay Torch của Colab** để tránh lỗi wheel trên runtime mới, nhưng code mô hình và toàn bộ hyperparameter RoBiS vẫn lấy từ đúng commit đã khóa.


In [2]:
import subprocess
import sys
import time
from pathlib import Path

REPO = Path("/content/RoBiS")
REPO_URL = "https://github.com/xrli-U/RoBiS.git"

def run(cmd, cwd=None, retries=1):
    cmd = [str(x) for x in cmd]

    for attempt in range(1, retries + 1):
        print("$", " ".join(cmd))

        p = subprocess.run(
            cmd,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )

        if p.stdout:
            print(p.stdout)

        if p.returncode == 0:
            return p

        if attempt < retries:
            print(f"Retry {attempt}/{retries}...")
            time.sleep(3)
        else:
            raise RuntimeError(
                f"Command failed with exit code {p.returncode}:\n"
                + " ".join(cmd)
            )


# ============================================================
# 1. Kiểm tra GPU
# ============================================================

archive = ARCHIVE_DIR / f"{CATEGORY}.tar.gz"
assert archive.is_file(), f"Không tìm thấy: {archive}"

run(["nvidia-smi"])


# ============================================================
# 2. KHÔNG XÓA repo nếu đã tồn tại
# ============================================================

if (REPO / ".git").exists():
    print(f"Reuse existing repo: {REPO}")

else:
    if REPO.exists():
        raise RuntimeError(
            f"{REPO} tồn tại nhưng không phải Git repo.\n"
            "Không tự động xóa để tránh mất dữ liệu."
        )

    run(
        [
            "git",
            "-c", "http.version=HTTP/1.1",
            "clone",
            "--filter=blob:none",
            "--no-checkout",
            REPO_URL,
            str(REPO),
        ],
        retries=3,
    )


# ============================================================
# 3. Fetch CHÍNH XÁC commit RoBiS cần dùng
# ============================================================

run(
    [
        "git",
        "-c", "http.version=HTTP/1.1",
        "fetch",
        "--depth", "1",
        "origin",
        ROBIS_COMMIT,
    ],
    cwd=REPO,
    retries=3,
)


# ============================================================
# 4. Checkout commit
# ============================================================

run(
    [
        "git",
        "checkout",
        "--detach",
        ROBIS_COMMIT,
    ],
    cwd=REPO,
)


# ============================================================
# 5. Verify
# ============================================================

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

assert actual_commit == ROBIS_COMMIT, (
    actual_commit,
    ROBIS_COMMIT,
)

print("REPO PASS")
print("Commit:", actual_commit)
# ============================================================
# 6. Install RoBiS dependencies
# ============================================================

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "timm==0.9.16",
    "kornia==0.7.1",
    "adeval==1.1.0",
    "tabulate==0.9.0",
    "tifffile>=2023.7",
    "scikit-image>=0.21",
    "scikit-learn>=1.3",
    "opencv-python-headless>=4.8",
])

import importlib
importlib.invalidate_caches()

import kornia
import timm
import adeval

print("DEPENDENCIES PASS")
print("Kornia:", kornia.__version__)
print("timm  :", timm.__version__)

$ nvidia-smi
Thu Sep 24 03:24:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   59C    P8             14W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------------

/usr/local/lib/python3.13/dist-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


DEPENDENCIES PASS
Kornia: 0.7.1
timm  : 0.9.16


## 3. Cache DINOv2-R weights trên Drive

RoBiS tự tải DINOv2-R vào `backbones/weights`. Cell này biến thư mục đó thành symlink tới Drive để đổi category / restart runtime không phải tải lại.


In [3]:
BACKBONE_CACHE = WEIGHT_CACHE / "backbones"
BACKBONE_CACHE.mkdir(parents=True, exist_ok=True)

local_weights = REPO / "backbones" / "weights"

if local_weights.is_symlink() or local_weights.is_file():
    local_weights.unlink()
elif local_weights.exists():
    shutil.rmtree(local_weights)

os.symlink(
    BACKBONE_CACHE,
    local_weights,
    target_is_directory=True,
)

print("DINO cache:", BACKBONE_CACHE)


DINO cache: /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/robis_weight_cache/backbones


## 4. Extract **chỉ category đang chọn**

Notebook tìm:

```text
data/<CATEGORY>.tar.gz
```

Các split được lấy nếu có:
- `train`
- `validation`
- `test_public`
- `test_private`
- `test_private_mixed`

Không giải nén 7 category còn lại.


In [4]:
SPLITS_NEEDED = {
    "train",
    "validation",
    "test_public",
    "test_private",
    "test_private_mixed",
}

archive = ARCHIVE_DIR / f"{CATEGORY}.tar.gz"
assert archive.is_file(), f"Không tìm thấy: {archive}"

category_native_root = MAD2_NATIVE_ROOT / CATEGORY

def split_has_png(root: Path, split: str):
    p = root / split
    return p.exists() and any(p.rglob("*.png"))

def extract_selected_category(
    archive_path: Path,
    destination: Path,
    force=False,
):
    destination.mkdir(parents=True, exist_ok=True)

    required = {
        "train",
        "test_public",
        "test_private",
        "test_private_mixed",
    }

    if not force and all(split_has_png(destination, s) for s in required):
        print("[SKIP EXTRACT] đã đủ split bắt buộc:", destination)
        return

    for split in SPLITS_NEEDED:
        shutil.rmtree(destination / split, ignore_errors=True)

    counts = {s: 0 for s in SPLITS_NEEDED}

    with tarfile.open(archive_path, "r:gz") as tf:
        for member in tf.getmembers():
            parts = PurePosixPath(member.name).parts
            hits = [i for i, p in enumerate(parts) if p in SPLITS_NEEDED]
            if not hits:
                continue

            i = hits[0]
            split = parts[i]
            rel_parts = parts[i:]

            if any(p in ("", ".", "..") for p in rel_parts):
                raise RuntimeError(f"Unsafe archive path: {member.name}")

            target = destination.joinpath(*rel_parts)

            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
                continue

            if not member.isfile():
                continue

            target.parent.mkdir(parents=True, exist_ok=True)

            src = tf.extractfile(member)
            if src is None:
                raise RuntimeError(member.name)

            with src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)

            counts[split] += 1

    for split in required:
        assert split_has_png(destination, split), f"Thiếu {CATEGORY}/{split}"

    print("Extract counts:", counts)

extract_selected_category(
    archive,
    category_native_root,
    force=FORCE_EXTRACT,
)

# Audit native data
audit_native = {
    "train/good": len(list((category_native_root / "train" / "good").glob("*.png"))),
    "test_public/good": len(list((category_native_root / "test_public" / "good").glob("*.png"))),
    "test_public/bad": len(list((category_native_root / "test_public" / "bad").glob("*.png"))),
    "test_public/gt_bad": len(list(
        (category_native_root / "test_public" / "ground_truth" / "bad").glob("*_mask.png")
    )),
    "test_private": len(list((category_native_root / "test_private").glob("*.png"))),
    "test_private_mixed": len(list((category_native_root / "test_private_mixed").glob("*.png"))),
}

for k, v in audit_native.items():
    print(f"{k:32s}: {v}")

for k, v in audit_native.items():
    assert v > 0, f"Thiếu dữ liệu: {k}"

print("NATIVE DATA PASS:", CATEGORY)


Extract counts: {'test_private_mixed': 321, 'test_private': 321, 'test_public': 252, 'validation': 46, 'train': 412}
train/good                      : 412
test_public/good                : 72
test_public/bad                 : 90
test_public/gt_bad              : 90
test_private                    : 321
test_private_mixed              : 321
NATIVE DATA PASS: can


## 5. Swin-Cropping — đúng repo RoBiS

Repo dùng:
- window = **1024**
- overlap = **0.1**
- zero-padding ở biên

Dữ liệu crop chỉ nằm local `/content`, không ghi hàng chục GB lên Drive.


In [5]:
def processed_ready(category: str):
    root = PROCESSED_ROOT / category
    checks = [
        root / "train" / "good",
        root / "test_public" / "good",
        root / "test_public" / "bad",
        root / "test_public" / "ground_truth" / "bad",
        root / "test_private",
        root / "test_private_mixed",
    ]
    return all(p.exists() and any(p.rglob("*.png")) for p in checks)

if FORCE_CROP:
    shutil.rmtree(PROCESSED_ROOT / CATEGORY, ignore_errors=True)

if not processed_ready(CATEGORY):
    run([
        sys.executable,
        "swin-cropping.py",
        "--data_path", MAD2_NATIVE_ROOT,
        "--save_path", PROCESSED_ROOT,
    ], cwd=REPO)

assert processed_ready(CATEGORY)

processed_counts = {
    "train/good": len(list((PROCESSED_ROOT / CATEGORY / "train" / "good").glob("*.png"))),
    "public/good": len(list((PROCESSED_ROOT / CATEGORY / "test_public" / "good").glob("*.png"))),
    "public/bad": len(list((PROCESSED_ROOT / CATEGORY / "test_public" / "bad").glob("*.png"))),
    "private": len(list((PROCESSED_ROOT / CATEGORY / "test_private").glob("*.png"))),
    "private_mixed": len(list((PROCESSED_ROOT / CATEGORY / "test_private_mixed").glob("*.png"))),
}

print("PROCESSED DATA:")
for k, v in processed_counts.items():
    print(f"{k:24s}: {v}")

print("SWIN-CROPPING PASS:", CATEGORY)


$ /usr/bin/python3 swin-cropping.py --data_path /content/RoBiS/data/mvtec_ad_2 --save_path /content/RoBiS/mvtec_ad_2_processed
sheet_metal processing...
vial processing...
wallplugs processing...
walnuts processing...
can processing...
can finished.
fabric processing...
fruit_jelly processing...
rice processing...

PROCESSED DATA:
train/good              : 1236
public/good             : 216
public/bad              : 270
private                 : 963
private_mixed           : 963
SWIN-CROPPING PASS: can


## 6. Import RoBiS modules + model builder

**Quan trọng:** training transform dùng `bright_aug=True`, tức đúng augmentation trong `dataset.py` của RoBiS:
- xác suất 0.5 áp dụng exposure ngẫu nhiên khoảng `[-0.2, 0.2]`
- cộng Gaussian noise `σ = 15`


In [6]:
from pathlib import Path
import sys
import os
import importlib
import warnings

REPO = Path("/content/RoBiS")
assert REPO.exists(), f"Không thấy repo: {REPO}"

layers_dir = REPO / "dinov2" / "layers"
assert layers_dir.exists(), f"Không thấy: {layers_dir}"

init_file = layers_dir / "__init__.py"

init_file.write_text(
"""from .attention import Attention, MemEffAttention
from .block import Block, NestedTensorBlock
from .drop_path import DropPath
from .layer_scale import LayerScale
from .mlp import Mlp
from .patch_embed import PatchEmbed
from .swiglu_ffn import SwiGLUFFN, SwiGLUFFNFused

__all__ = [
    "Attention",
    "MemEffAttention",
    "Block",
    "NestedTensorBlock",
    "DropPath",
    "LayerScale",
    "Mlp",
    "PatchEmbed",
    "SwiGLUFFN",
    "SwiGLUFFNFused",
]
""",
    encoding="utf-8"
)

print("Đã tạo:", init_file)

# Xóa CHỈ cache import dinov2 bị lỗi.
# Không xóa model, checkpoint, dataset hay kết quả đã chạy.
for name in list(sys.modules.keys()):
    if name == "dinov2" or name.startswith("dinov2."):
        del sys.modules[name]

importlib.invalidate_caches()

os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

warnings.filterwarnings(
    "ignore",
    message=r".*torch\.cuda\.amp\.custom_fwd.*",
    category=FutureWarning,
)

from dinov2.layers import (
    Mlp,
    PatchEmbed,
    SwiGLUFFNFused,
    MemEffAttention,
    Block,
)

print("DINOv2 layers import: PASS")

Đã tạo: /content/RoBiS/dinov2/layers/__init__.py
DINOv2 layers import: PASS


In [7]:

os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import tifffile
import math

from functools import partial
from tqdm.auto import tqdm
from PIL import Image
from torch.nn.init import trunc_normal_
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Compatibility với code repo cũ.
if not hasattr(tifffile, "imsave"):
    tifffile.imsave = tifffile.imwrite

from optimizers import StableAdamW

from utils import (
    WarmCosineScheduler,
    global_cosine_hm_adaptive,
    setup_seed,
    cal_anomaly_maps,
    get_gaussian_kernel,
)

from dataset import (
    MVTec2Dataset,
    get_data_transforms,
)

from models import vit_encoder
from models.uad import INP_Former

from models.vision_transformer import (
    Mlp,
    Aggregation_Block,
    Prototype_Block,
)

from merging import (
    reconstruct_image,
    resolution_dict,
)

from MEBin import MEBin

assert torch.cuda.is_available(), "Bật GPU runtime trước khi chạy."

device = torch.device("cuda:0")
setup_seed(SEED)

data_transform, gt_transform = get_data_transforms(
    INPUT_SIZE,
    CROP_SIZE,
)

data_train_transform, _ = get_data_transforms(
    INPUT_SIZE,
    CROP_SIZE,
    bright_aug=True,
)

def build_model():
    target_layers = [2, 3, 4, 5, 6, 7, 8, 9]
    fuse_layer_encoder = [[0, 1, 2, 3], [4, 5, 6, 7]]
    fuse_layer_decoder = [[0, 1, 2, 3], [4, 5, 6, 7]]

    encoder = vit_encoder.load("dinov2reg_vit_base_14")

    embed_dim = 768
    num_heads = 12

    bottleneck = nn.ModuleList([
        Mlp(
            embed_dim,
            embed_dim * 4,
            embed_dim,
            drop=0.0,
        )
    ])

    inp = nn.ParameterList([
        nn.Parameter(torch.randn(INP_NUM, embed_dim))
    ])

    extractor = nn.ModuleList([
        Aggregation_Block(
            dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=4.0,
            qkv_bias=True,
            norm_layer=partial(nn.LayerNorm, eps=1e-8),
        )
    ])

    decoder = nn.ModuleList([
        Prototype_Block(
            dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=4.0,
            qkv_bias=True,
            norm_layer=partial(nn.LayerNorm, eps=1e-8),
        )
        for _ in range(8)
    ])

    model = INP_Former(
        encoder=encoder,
        bottleneck=bottleneck,
        aggregation=extractor,
        decoder=decoder,
        target_layers=target_layers,
        remove_class_token=True,
        fuse_layer_encoder=fuse_layer_encoder,
        fuse_layer_decoder=fuse_layer_decoder,
        prototype_token=inp,
    ).to(device)

    trainable = nn.ModuleList([
        bottleneck,
        decoder,
        extractor,
        inp,
    ])

    # Official RoBiS initialization
    for module in trainable.modules():
        if isinstance(module, nn.Linear):
            trunc_normal_(
                module.weight,
                std=0.01,
                a=-0.03,
                b=0.03,
            )
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

        elif isinstance(module, nn.LayerNorm):
            nn.init.constant_(module.bias, 0)
            nn.init.constant_(module.weight, 1.0)

    return model, trainable

print("GPU       :", torch.cuda.get_device_name(0))
print("Torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
print("Category  :", CATEGORY)
print("Model     : DINOv2-R ViT-B/14 + INP-Former")
print("Input     :", INPUT_SIZE)


GPU       : NVIDIA L4
Torch     : 2.11.0+cu128
CUDA      : 12.8
Category  : can
Model     : DINOv2-R ViT-B/14 + INP-Former
Input     : 518


## 7. Train selected category — có resume

Thuật toán train giữ nguyên repo:
\[
L = L_{global\_cosine\_hm} + 0.2 L_g
\]

- `global_cosine_hm_adaptive(..., y=3)`
- gradient clipping `0.1`
- `StableAdamW`
- 200 epochs

Notebook thêm `last.pth` để Colab disconnect vẫn resume được. `model.pth` là checkpoint cuối cùng, tương thích đúng state dict của repo.


In [ ]:
SAVE_NAME = (
    "INP-Former-Single-Class"
    "_dataset=MVTec-AD-2"
    "_Encoder=dinov2reg_vit_base_14"
    f"_Resize={INPUT_SIZE}"
    f"_Crop={CROP_SIZE}"
    f"_INP_num={INP_NUM}"
)

CHECKPOINT_ROOT = DRIVE_OUT / "saved_weights" / SAVE_NAME
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for data in iter(lambda: f.read(chunk), b""):
            h.update(data)
    return h.hexdigest()

def capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])

def optimizer_to(optimizer, device):
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(device)

def train_selected_category(category, force=False):
    out_dir = CHECKPOINT_ROOT / category
    out_dir.mkdir(parents=True, exist_ok=True)

    final_ckpt = out_dir / "model.pth"
    resume_ckpt = out_dir / "last.pth"
    history_path = out_dir / "train_history.csv"

    if final_ckpt.exists() and not force:
        print("[SKIP TRAIN] final model exists:", final_ckpt)
        return final_ckpt

    setup_seed(SEED)

    train_data = ImageFolder(
        root=str(PROCESSED_ROOT / category / "train"),
        transform=data_train_transform,
    )

    train_loader = DataLoader(
        train_data,
        batch_size=TRAIN_BATCH,
        shuffle=True,
        num_workers=SAFE_NUM_WORKERS,
        pin_memory=True,
        drop_last=True,

        persistent_workers=(SAFE_NUM_WORKERS > 0),
        prefetch_factor=2 if SAFE_NUM_WORKERS > 0 else None,
    )

    assert len(train_loader) > 0, "Train set nhỏ hơn batch size hoặc dữ liệu lỗi."

    model, trainable = build_model()

    optimizer = StableAdamW(
        [{"params": trainable.parameters()}],
        lr=1e-3,
        betas=(0.9, 0.999),
        weight_decay=1e-4,
        amsgrad=True,
        eps=1e-10,
    )

    scheduler = WarmCosineScheduler(
        optimizer,
        base_value=1e-3,
        final_value=1e-4,
        total_iters=EPOCHS * len(train_loader),
        warmup_iters=100,
    )

    start_epoch = 0
    history = []

    if history_path.exists() and not force:
        try:
            history = pd.read_csv(history_path).to_dict(orient="records")
        except Exception as e:
            print("[WARN] train_history:", e)

    if resume_ckpt.exists() and not force:
        print("[RESUME]", resume_ckpt)
        state = torch.load(
            resume_ckpt,
            map_location="cpu",
            weights_only=False,
        )

        model.load_state_dict(state["model"], strict=True)
        optimizer.load_state_dict(state["optimizer"])
        optimizer_to(optimizer, device)
        scheduler.load_state_dict(state["scheduler"])

        start_epoch = int(state["epoch"]) + 1
        restore_rng_state(state.get("rng_state"))

        print(f"Resume epoch {start_epoch}/{EPOCHS}")

    print("=" * 72)
    print("TRAIN CATEGORY :", category)
    print("Train crops    :", len(train_data))
    print("Epochs         :", EPOCHS)
    print("Start epoch    :", start_epoch)
    print("Batch          :", TRAIN_BATCH)
    print("Batches/epoch  :", len(train_loader))
    print("=" * 72)

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        losses = []

        pbar = tqdm(
            train_loader,
            desc=f"{category} {epoch+1}/{EPOCHS}",
            leave=False,
        )

        for image, _ in pbar:
            image = image.to(device, non_blocking=True)

            en, de, g_loss = model(image)

            loss = global_cosine_hm_adaptive(
                en,
                de,
                y=3,
            )
            loss = loss + 0.2 * g_loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()

            nn.utils.clip_grad_norm_(
                trainable.parameters(),
                max_norm=0.1,
            )

            optimizer.step()
            scheduler.step()

            losses.append(float(loss.item()))
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        mean_loss = float(np.mean(losses))

        history = [
            row for row in history
            if int(row["epoch"]) != epoch + 1
        ]
        history.append({
            "epoch": epoch + 1,
            "loss": mean_loss,
        })

        pd.DataFrame(history).sort_values("epoch").to_csv(
            history_path,
            index=False,
        )

        print(
            f"{category}: epoch {epoch+1:03d}/{EPOCHS} "
            f"loss={mean_loss:.6f}"
        )

        if ((epoch + 1) % SAVE_EVERY == 0) or (epoch + 1 == EPOCHS):
            torch.save(
                {
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "rng_state": capture_rng_state(),
                    "category": category,
                    "repo_commit": ROBIS_COMMIT,
                    "input_size": INPUT_SIZE,
                    "crop_size": CROP_SIZE,
                    "inp_num": INP_NUM,
                    "epochs": EPOCHS,
                },
                resume_ckpt,
            )

    torch.save(model.state_dict(), final_ckpt)

    metadata = {
        "category": category,
        "repo_commit": ROBIS_COMMIT,
        "encoder": "dinov2reg_vit_base_14",
        "input_size": INPUT_SIZE,
        "crop_size": CROP_SIZE,
        "inp_num": INP_NUM,
        "epochs": EPOCHS,
        "train_batch": TRAIN_BATCH,
        "seed": SEED,
        "swin_window": SWIN_WINDOW,
        "swin_overlap": SWIN_OVERLAP,
        "train_crops": len(train_data),
        "checkpoint_sha256": sha256_file(final_ckpt),
    }

    (out_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )

    del model, trainable, optimizer, scheduler, train_loader
    torch.cuda.empty_cache()

    print("[TRAIN COMPLETE]", final_ckpt)
    return final_ckpt

checkpoint = train_selected_category(
    CATEGORY,
    force=FORCE_TRAIN,
)

print("CHECKPOINT:", checkpoint)


[RESUME] /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/robis_results_single_class/saved_weights/INP-Former-Single-Class_dataset=MVTec-AD-2_Encoder=dinov2reg_vit_base_14_Resize=518_Crop=518_INP_num=6/can/last.pth
Resume epoch 50/200
TRAIN CATEGORY : can
Train crops    : 1236
Epochs         : 200
Start epoch    : 50
Batch          : 16
Batches/epoch  : 77


can 51/200:   0%|          | 0/77 [00:00<?, ?it/s]

can: epoch 051/200 loss=0.167432


can 52/200:   0%|          | 0/77 [00:00<?, ?it/s]

can: epoch 052/200 loss=0.166870


...

can 109/200:   0%|          | 0/77 [00:00<?, ?it/s]

can: epoch 109/200 loss=0.151387


can 110/200:   0%|          | 0/77 [00:00<?, ?it/s]

can: epoch 110/200 loss=0.151428


can 111/200:   0%|          | 0/77 [00:00<?, ?it/s]

## 8. Inference continuous maps — streaming nhưng giữ đúng logic repo

Repo gốc gom toàn bộ crop maps rồi:
1. resize về `256×256`,
2. Gaussian smoothing,
3. **global min-max normalization trên toàn bộ crop maps của category/split** về `[0, 255]`,
4. ghi `float16 TIFF`.

Cell dưới làm cùng phép toán nhưng streaming:
- pass 1: infer và lưu raw crop map local,
- lấy global min/max,
- pass 2: normalize `[0,255]` và ghi TIFF.

Cách này giảm VRAM đáng kể nhưng không đổi score map toán học.


In [ ]:
gaussian_kernel = get_gaussian_kernel(
    kernel_size=5,
    sigma=4,
).to(device)

def load_trained_model(category):
    ckpt = CHECKPOINT_ROOT / category / "model.pth"
    assert ckpt.is_file(), f"Thiếu checkpoint: {ckpt}"

    model, _ = build_model()
    state = torch.load(
        ckpt,
        map_location="cpu",
        weights_only=True,
    )
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

def make_test_dataset(category, split):
    return MVTec2Dataset(
        root=str(PROCESSED_ROOT / category),
        transform=data_transform,
        gt_transform=gt_transform,
        phase="test",
        test_type=split,
    )

@torch.inference_mode()
def predict_crop_batch(model, images):
    images = images.to(device, non_blocking=True)
    en, de, _ = model(images)

    anomaly_map, _ = cal_anomaly_maps(
        en,
        de,
        images.shape[-1],
    )

    anomaly_map = F.interpolate(
        anomaly_map,
        size=MAP_SIZE,
        mode="bilinear",
        align_corners=False,
    )

    anomaly_map = gaussian_kernel(anomaly_map)
    return anomaly_map[:, 0]

def infer_split_to_patch_tiffs(
    category,
    split,
    force=False,
):
    if split == "test_public":
        patch_root = (
            LOCAL_AMAP_ROOT
            / "anomaly_images_public"
            / category
            / split
        )
    else:
        patch_root = (
            LOCAL_AMAP_ROOT
            / "anomaly_images"
            / category
            / split
        )

    if force:
        shutil.rmtree(patch_root, ignore_errors=True)

    shutil.rmtree(patch_root, ignore_errors=True)
    patch_root.mkdir(parents=True, exist_ok=True)

    raw_tmp = LOCAL_RUN / "raw_patch_tmp" / category / split
    shutil.rmtree(raw_tmp, ignore_errors=True)
    raw_tmp.mkdir(parents=True, exist_ok=True)

    dataset = make_test_dataset(category, split)

    loader = DataLoader(
        dataset,
        batch_size=INFER_BATCH,
        shuffle=False,
        num_workers=SAFE_NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(SAFE_NUM_WORKERS > 0),
        prefetch_factor=2 if SAFE_NUM_WORKERS > 0 else None,
    )

    model = load_trained_model(category)

    records = []
    global_min = np.inf
    global_max = -np.inf

    # --------------------------------------------------------
    # PASS 1 — raw score maps
    # --------------------------------------------------------
    for images, _, _, img_paths in tqdm(
        loader,
        desc=f"INFER {category}/{split}",
    ):
        maps = (
            predict_crop_batch(model, images)
            .float()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        for amap, img_path in zip(maps, img_paths):
            src = Path(img_path)

            global_min = min(global_min, float(amap.min()))
            global_max = max(global_max, float(amap.max()))

            rel_group = src.parent.name if split == "test_public" else None

            tmp_path = raw_tmp / f"{len(records):08d}.npy"
            np.save(tmp_path, amap)

            records.append({
                "src": str(src),
                "group": rel_group,
                "tmp": str(tmp_path),
            })

    if not np.isfinite(global_min) or not np.isfinite(global_max):
        raise RuntimeError("Không thu được anomaly map.")

    if global_max <= global_min:
        raise RuntimeError(
            f"Degenerate anomaly score range: {global_min}..{global_max}"
        )

    # --------------------------------------------------------
    # PASS 2 — global normalize đúng evaluation_batch của repo
    # --------------------------------------------------------
    for row in tqdm(records, desc=f"NORMALIZE {category}/{split}"):
        src = Path(row["src"])
        amap = np.load(row["tmp"]).astype(np.float32)

        amap = (
            (amap - global_min)
            / (global_max - global_min)
            * 255.0
        )

        if split == "test_public":
            dst = patch_root / row["group"] / f"{src.stem}.tiff"
        else:
            dst = patch_root / f"{src.stem}.tiff"

        dst.parent.mkdir(parents=True, exist_ok=True)

        tifffile.imwrite(
            dst,
            amap.astype(np.float16),
        )

    del model, loader
    torch.cuda.empty_cache()
    shutil.rmtree(raw_tmp, ignore_errors=True)

    print("Patch TIFF root:", patch_root)
    print("Global score min/max:", global_min, global_max)

    return patch_root

def merge_selected_category(category, split, patch_root):
    if split == "test_public":
        for label in sorted(
            p.name for p in patch_root.iterdir()
            if p.is_dir()
        ):
            reconstruct_image(
                str(patch_root / label),
                category,
                window_size=MAP_SIZE,
                desired_overlap=SWIN_OVERLAP,
            )
    else:
        reconstruct_image(
            str(patch_root),
            category,
            window_size=MAP_SIZE,
            desired_overlap=SWIN_OVERLAP,
        )

    print("MERGE COMPLETE:", category, split)

def copy_merged_public_to_drive(category, patch_root):
    dst = CATEGORY_OUT / "public_continuous"

    if dst.exists():
        shutil.rmtree(dst)

    shutil.copytree(
        patch_root,
        dst,
    )

    print("Public continuous maps:", dst)
    return dst

PUBLIC_CONT_DIR = CATEGORY_OUT / "public_continuous"

if RUN_PUBLIC_EVAL:
    expected_public = (
        audit_native["test_public/good"]
        + audit_native["test_public/bad"]
    )

    existing = (
        len(list(PUBLIC_CONT_DIR.rglob("*.tiff")))
        if PUBLIC_CONT_DIR.exists()
        else 0
    )

    if existing == expected_public and not FORCE_PUBLIC_INFERENCE:
        print("[SKIP PUBLIC INFERENCE] merged maps already on Drive:", PUBLIC_CONT_DIR)
    else:
        public_patch_root = infer_split_to_patch_tiffs(
            CATEGORY,
            "test_public",
            force=FORCE_PUBLIC_INFERENCE,
        )

        merge_selected_category(
            CATEGORY,
            "test_public",
            public_patch_root,
        )

        PUBLIC_CONT_DIR = copy_merged_public_to_drive(
            CATEGORY,
            public_patch_root,
        )

    print("Public merged TIFFs:", len(list(PUBLIC_CONT_DIR.rglob("*.tiff"))))


## 9. AU-PRO@0.05 + pixel F1-max trên `test_public`

- **AU-PRO@0.05** là metric threshold-independent chính mà bạn đang theo dõi.
- `pixel_F1_max` chỉ dùng **diagnostic / reproduction ablation** trên public vì nó chọn threshold tối ưu từ GT public.
- Không được dùng threshold này để sinh private submission.


In [ ]:
MAX_FPR = 0.05

def compute_aupro_005(normal_scores, region_scores):
    normal_scores = np.concatenate(normal_scores).astype(
        np.float32,
        copy=False,
    )

    num_regions = len(region_scores)
    if num_regions == 0:
        raise RuntimeError("Không có GT connected component.")

    anomaly_scores = []
    anomaly_weights = []

    for scores in region_scores:
        scores = np.asarray(scores, dtype=np.float32).reshape(-1)
        if scores.size == 0:
            continue

        anomaly_scores.append(scores)
        anomaly_weights.append(
            np.full(
                scores.size,
                1.0 / (num_regions * scores.size),
                dtype=np.float64,
            )
        )

    anomaly_scores = np.concatenate(anomaly_scores)
    anomaly_weights = np.concatenate(anomaly_weights)

    num_normal = normal_scores.size
    if num_normal == 0:
        raise RuntimeError("Không có normal pixels.")

    scores_all = np.concatenate([
        normal_scores,
        anomaly_scores,
    ])

    fpr_delta = np.concatenate([
        np.full(
            num_normal,
            1.0 / num_normal,
            dtype=np.float64,
        ),
        np.zeros(
            anomaly_scores.size,
            dtype=np.float64,
        ),
    ])

    pro_delta = np.concatenate([
        np.zeros(
            num_normal,
            dtype=np.float64,
        ),
        anomaly_weights,
    ])

    order = np.argsort(
        scores_all,
        kind="mergesort",
    )[::-1]

    sorted_scores = scores_all[order]
    cumulative_fpr = np.cumsum(fpr_delta[order])
    cumulative_pro = np.cumsum(pro_delta[order])

    group_ends = np.r_[
        np.flatnonzero(
            sorted_scores[1:] != sorted_scores[:-1]
        ),
        sorted_scores.size - 1,
    ]

    fpr = np.r_[0.0, cumulative_fpr[group_ends]]
    pro = np.r_[0.0, cumulative_pro[group_ends]]

    unique_fpr, first_idx, counts = np.unique(
        fpr,
        return_index=True,
        return_counts=True,
    )

    unique_pro = pro[first_idx + counts - 1]

    idx = np.searchsorted(
        unique_fpr,
        MAX_FPR,
        side="left",
    )

    if idx >= len(unique_fpr):
        raise RuntimeError("FPR curve không đạt 0.05.")

    if np.isclose(unique_fpr[idx], MAX_FPR):
        x = unique_fpr[:idx + 1]
        y = unique_pro[:idx + 1]
    else:
        x0, x1 = unique_fpr[idx - 1], unique_fpr[idx]
        y0, y1 = unique_pro[idx - 1], unique_pro[idx]

        y_at = y0 + (
            (MAX_FPR - x0)
            * (y1 - y0)
            / (x1 - x0)
        )

        x = np.r_[unique_fpr[:idx], MAX_FPR]
        y = np.r_[unique_pro[:idx], y_at]

    area = np.trapezoid(y, x)
    return float(area / MAX_FPR)

def load_public_gt_native(category, kind, stem):
    native_root = MAD2_NATIVE_ROOT / category / "test_public"

    if kind == "good":
        image_path = native_root / "good" / f"{stem}.png"
        gt = None
    else:
        image_path = native_root / "bad" / f"{stem}.png"
        gt_path = (
            native_root
            / "ground_truth"
            / "bad"
            / f"{stem}_mask.png"
        )
        assert gt_path.is_file(), gt_path
        gt = np.array(Image.open(gt_path).convert("L")) > 0

    assert image_path.is_file(), image_path
    return image_path, gt

def evaluate_public_continuous(category, continuous_dir):
    normal_scores = []
    region_scores = []

    all_gt_flat = []
    all_score_flat = []

    n_good = 0
    n_bad = 0

    for kind in ["good", "bad"]:
        paths = sorted((continuous_dir / kind).glob("*.tiff"))

        for map_path in tqdm(paths, desc=f"AUPRO {category}/{kind}"):
            amap = tifffile.imread(map_path).astype(np.float32)

            if kind == "good":
                gt = np.zeros_like(amap, dtype=np.uint8)
                n_good += 1

            else:
                _, gt_native = load_public_gt_native(
                    category,
                    kind,
                    map_path.stem,
                )

                gt = cv2.resize(
                    gt_native.astype(np.uint8),
                    (amap.shape[1], amap.shape[0]),
                    interpolation=cv2.INTER_NEAREST,
                )

                gt = (gt > 0).astype(np.uint8)
                n_bad += 1

            normal_scores.append(
                amap[gt == 0].reshape(-1)
            )

            if gt.any():
                num_labels, labels = cv2.connectedComponents(
                    gt,
                    connectivity=8,
                )

                for region_id in range(1, num_labels):
                    region_scores.append(
                        amap[labels == region_id].reshape(-1)
                    )

            all_gt_flat.append(gt.reshape(-1))
            all_score_flat.append(amap.reshape(-1))

    aupro = compute_aupro_005(
        normal_scores,
        region_scores,
    )

    # F1-max diagnostic only
    from sklearn.metrics import precision_recall_curve

    y_true = np.concatenate(all_gt_flat)
    y_score = np.concatenate(all_score_flat)

    precision, recall, thresholds = precision_recall_curve(
        y_true,
        y_score,
    )

    f1 = (
        2 * precision * recall
        / (precision + recall + 1e-12)
    )

    pixel_f1_max = float(
        np.nanmax(f1)
    )

    return {
        "Category": category,
        "AU_PRO_0.05": aupro,
        "AU_PRO_0.05_pct": aupro * 100.0,
        "pixel_F1_max": pixel_f1_max,
        "pixel_F1_max_pct": pixel_f1_max * 100.0,
        "good_images": n_good,
        "bad_images": n_bad,
        "gt_regions": len(region_scores),
    }

if RUN_PUBLIC_EVAL:
    continuous_metrics = evaluate_public_continuous(
        CATEGORY,
        PUBLIC_CONT_DIR,
    )

    display(pd.DataFrame([continuous_metrics]))

    print(
        f"{CATEGORY} AU-PRO@0.05 = "
        f"{continuous_metrics['AU_PRO_0.05_pct']:.4f}%"
    )


## 10. MEBin + mean+3std — đúng hậu xử lý RoBiS

RoBiS không dùng một threshold thủ công cho từng category. Binary mask thô là:

```text
MEBin mask  OR  mean+3std mask
```

MEBin tìm khoảng threshold ổn định theo số connected components, sau đó erosion `6×6`, 1 iteration.


In [ ]:
PUBLIC_COARSE_DIR = CATEGORY_OUT / "public_binary_coarse"

def binarize_selected_category(
    category,
    continuous_dir,
    output_dir,
):
    if output_dir.exists():
        shutil.rmtree(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    # Giữ thứ tự giống binarization.py:
    # anomaly_types = sorted(os.listdir(...))
    groups = sorted([
        p.name for p in continuous_dir.iterdir()
        if p.is_dir()
    ])

    amap_paths = []
    group_counts = []

    for group in groups:
        paths = sorted((continuous_dir / group).glob("*.tiff"))
        amap_paths.extend([str(p) for p in paths])
        group_counts.append(len(paths))

    assert amap_paths, "Không có continuous maps."

    mebin = MEBin(
        amap_paths,
        class_name=category,
    )

    binarized_maps, thresholds = mebin.binarize_anomaly_maps()

    start = 0
    rows = []

    for group, count in zip(groups, group_counts):
        group_out = output_dir / group
        group_out.mkdir(parents=True, exist_ok=True)

        for j in range(count):
            idx = start + j

            src = Path(amap_paths[idx])
            dst = group_out / f"{src.stem}.png"

            cv2.imwrite(
                str(dst),
                binarized_maps[idx],
            )

            rows.append({
                "group": group,
                "image": src.stem,
                "threshold_mebin": float(thresholds[idx]),
            })

        start += count

    pd.DataFrame(rows).to_csv(
        output_dir / "thresholds.csv",
        index=False,
    )

    print("COARSE BINARY:", output_dir)
    return output_dir

if RUN_PUBLIC_EVAL and RUN_MEBIN:
    PUBLIC_COARSE_DIR = binarize_selected_category(
        CATEGORY,
        PUBLIC_CONT_DIR,
        PUBLIC_COARSE_DIR,
    )


## 11. SAM-Finer — selected category

Đúng logic `SAM-Finer.py`:
- lấy **top-3 connected components** lớn nhất từ coarse mask,
- dùng bounding boxes làm box prompt cho SAM,
- `fabric`, `walnuts`: `multimask_output=True`,
- các class khác: `False`,
- fill holes sau refinement,
- `rice`: bỏ qua SAM-Finer.


In [ ]:
from scipy.ndimage import zoom
from segment_anything import sam_model_registry, SamPredictor

SAM_URLS = {
    "vit_h": (
        "sam_vit_h_4b8939.pth",
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",
    ),
    "vit_b": (
        "sam_vit_b_01ec64.pth",
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
    ),
}

def ensure_sam_checkpoint(variant):
    filename, url = SAM_URLS[variant]

    cache_path = WEIGHT_CACHE / filename
    if not cache_path.is_file():
        print("Downloading:", url)
        run(["wget", "-c", url, "-O", cache_path])

    local_path = REPO / filename

    if local_path.is_symlink() or local_path.is_file():
        local_path.unlink()
    elif local_path.exists():
        shutil.rmtree(local_path)

    os.symlink(cache_path, local_path)
    return local_path

def get_topn_bounding_boxes(binary_image, top_n=3):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_image
    )

    if num_labels <= 1:
        return []

    areas = stats[1:, cv2.CC_STAT_AREA]
    top_indices = np.argsort(areas)[-top_n:] + 1

    boxes = []
    for idx in top_indices:
        x = stats[idx, cv2.CC_STAT_LEFT]
        y = stats[idx, cv2.CC_STAT_TOP]
        w = stats[idx, cv2.CC_STAT_WIDTH]
        h = stats[idx, cv2.CC_STAT_HEIGHT]
        boxes.append([x, y, x + w, y + h])

    return boxes

def fill_holes(image):
    num_labels, labels = cv2.connectedComponents(image)
    mask = np.zeros_like(image)

    for label in range(1, num_labels):
        component_mask = (labels == label).astype(np.uint8)

        contours, _ = cv2.findContours(
            component_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )

        if contours:
            cv2.drawContours(
                mask,
                contours,
                -1,
                255,
                thickness=cv2.FILLED,
            )

    return cv2.bitwise_or(image, mask)

def samfiner_selected(
    category,
    split,
    coarse_root,
    final_root,
):
    if final_root.exists():
        shutil.rmtree(final_root)
    final_root.mkdir(parents=True, exist_ok=True)

    # Official RoBiS: rice không dùng SAM-Finer.
    if category == "rice":
        for src in coarse_root.rglob("*.png"):
            rel = src.relative_to(coarse_root)
            dst = final_root / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

        print("[RICE] Skip SAM-Finer, copied coarse masks.")
        return final_root

    variant = (
        "vit_b"
        if split == "test_private_mixed"
        else "vit_h"
    )

    checkpoint = ensure_sam_checkpoint(variant)

    sam = sam_model_registry[variant](
        checkpoint=str(checkpoint)
    )

    predictor = SamPredictor(
        sam.to(device)
    )

    multimask_tag = category in ["fabric", "walnuts"]

    native_split_root = (
        MAD2_NATIVE_ROOT
        / category
        / split
    )

    if split == "test_public":
        image_paths = (
            sorted((native_split_root / "bad").glob("*.png"))
            + sorted((native_split_root / "good").glob("*.png"))
        )
    else:
        image_paths = sorted(
            native_split_root.glob("*.png")
        )

    for image_path in tqdm(
        image_paths,
        desc=f"SAM {category}/{split}/{variant}",
    ):
        if split == "test_public":
            group = image_path.parent.name
            binary_path = (
                coarse_root
                / group
                / f"{image_path.stem}.png"
            )
            save_path = (
                final_root
                / group
                / f"{image_path.stem}.png"
            )
        else:
            binary_path = (
                coarse_root
                / f"{image_path.stem}.png"
            )
            save_path = (
                final_root
                / f"{image_path.stem}.png"
            )

        assert binary_path.is_file(), binary_path

        image = cv2.imread(str(image_path))
        binary_mask = cv2.imread(
            str(binary_path),
            cv2.IMREAD_GRAYSCALE,
        )

        # Giữ đúng code gốc: resize ảnh gốc về resolution binary mask.
        zoom_factors = (
            binary_mask.shape[0] / image.shape[0],
            binary_mask.shape[1] / image.shape[1],
            1,
        )

        image_small = zoom(
            image,
            zoom_factors,
            order=1,
        )

        bbox_list = get_topn_bounding_boxes(
            binary_mask,
            top_n=3,
        )

        if len(bbox_list) == 0:
            masks = np.zeros_like(binary_mask)

        else:
            predictor.set_image(image_small)
            mask_list = []

            for bbox in bbox_list:
                box_prompt = np.array(bbox)

                masks, _, _ = predictor.predict(
                    box=box_prompt,
                    return_logits=False,
                    multimask_output=multimask_tag,
                )

                masks = masks.sum(0)
                masks[masks > 0] = 255
                masks = masks.astype(np.uint8)
                mask_list.append(masks)

            masks = np.array(mask_list).max(0)
            masks = fill_holes(masks)

        save_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        cv2.imwrite(
            str(save_path),
            masks,
        )

    del predictor, sam
    torch.cuda.empty_cache()

    print("SAM-FINER OUTPUT:", final_root)
    return final_root

PUBLIC_FINAL_DIR = CATEGORY_OUT / "public_binary_final"

if RUN_PUBLIC_EVAL and RUN_MEBIN:
    if RUN_SAM_FINER:
        PUBLIC_FINAL_DIR = samfiner_selected(
            CATEGORY,
            "test_public",
            PUBLIC_COARSE_DIR,
            PUBLIC_FINAL_DIR,
        )
    else:
        if PUBLIC_FINAL_DIR.exists():
            shutil.rmtree(PUBLIC_FINAL_DIR)
        shutil.copytree(PUBLIC_COARSE_DIR, PUBLIC_FINAL_DIR)
        print("[SAM DISABLED] final = coarse")


## 12. SegF1 + ClassF1 trên `test_public`

`SegF1` dưới đây là **global pixel F1** trên toàn bộ public test của category:

\[
F1 = \frac{2TP}{2TP + FP + FN}
\]

GT được resize bằng nearest-neighbor về đúng kích thước binary prediction (RoBiS xuất map ở khoảng 1/4 native resolution).

`ClassF1` coi ảnh là anomaly nếu final binary mask có ít nhất một pixel anomaly.


In [ ]:
from sklearn.metrics import f1_score

def evaluate_public_binary(
    category,
    final_binary_dir,
):
    tp = fp = fn = tn = 0

    y_true_img = []
    y_pred_img = []

    image_rows = []

    for kind in ["good", "bad"]:
        pred_paths = sorted(
            (final_binary_dir / kind).glob("*.png")
        )

        for pred_path in tqdm(
            pred_paths,
            desc=f"SegF1 {category}/{kind}",
        ):
            pred = cv2.imread(
                str(pred_path),
                cv2.IMREAD_GRAYSCALE,
            )
            pred = pred > 0

            if kind == "good":
                gt = np.zeros_like(pred, dtype=bool)
                true_img = 0
            else:
                _, gt_native = load_public_gt_native(
                    category,
                    kind,
                    pred_path.stem,
                )

                gt = cv2.resize(
                    gt_native.astype(np.uint8),
                    (pred.shape[1], pred.shape[0]),
                    interpolation=cv2.INTER_NEAREST,
                ) > 0

                true_img = 1

            tp_i = int(np.logical_and(pred, gt).sum())
            fp_i = int(np.logical_and(pred, ~gt).sum())
            fn_i = int(np.logical_and(~pred, gt).sum())
            tn_i = int(np.logical_and(~pred, ~gt).sum())

            tp += tp_i
            fp += fp_i
            fn += fn_i
            tn += tn_i

            pred_img = int(pred.any())

            y_true_img.append(true_img)
            y_pred_img.append(pred_img)

            denom_i = 2 * tp_i + fp_i + fn_i
            f1_i = (
                2 * tp_i / denom_i
                if denom_i > 0
                else np.nan
            )

            image_rows.append({
                "kind": kind,
                "image": pred_path.stem,
                "TP": tp_i,
                "FP": fp_i,
                "FN": fn_i,
                "TN": tn_i,
                "SegF1_image": f1_i,
                "predicted_anomaly": pred_img,
            })

    seg_denom = 2 * tp + fp + fn

    seg_f1 = (
        2 * tp / seg_denom
        if seg_denom > 0
        else 0.0
    )

    class_f1 = f1_score(
        y_true_img,
        y_pred_img,
        zero_division=0,
    )

    per_image_df = pd.DataFrame(image_rows)

    return {
        "SegF1": float(seg_f1),
        "SegF1_pct": float(seg_f1 * 100.0),
        "ClassF1": float(class_f1),
        "ClassF1_pct": float(class_f1 * 100.0),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }, per_image_df

if RUN_PUBLIC_EVAL and RUN_MEBIN:
    binary_metrics, per_image_binary = evaluate_public_binary(
        CATEGORY,
        PUBLIC_FINAL_DIR,
    )

    final_public_metrics = {
        **continuous_metrics,
        **binary_metrics,
    }

    metrics_df = pd.DataFrame([
        final_public_metrics
    ])

    METRIC_DIR = DRIVE_OUT / "metrics"
    METRIC_DIR.mkdir(parents=True, exist_ok=True)

    metric_path = METRIC_DIR / f"{CATEGORY}.csv"
    metrics_df.to_csv(metric_path, index=False)

    per_image_binary.to_csv(
        CATEGORY_OUT / "public_binary_per_image.csv",
        index=False,
    )

    display(metrics_df)

    print(
        f"{CATEGORY}: "
        f"AU-PRO@0.05={final_public_metrics['AU_PRO_0.05_pct']:.3f}% | "
        f"SegF1={final_public_metrics['SegF1_pct']:.3f}% | "
        f"ClassF1={final_public_metrics['ClassF1_pct']:.3f}%"
    )

    print("Saved:", metric_path)


## 13. OPTIONAL — private / private-mixed → submission

Mặc định `RUN_PRIVATE_INFERENCE = False`.

Khi bật `True`, notebook chạy đúng chuỗi:
1. crop inference,
2. global normalization,
3. merge,
4. MEBin + mean+3std,
5. SAM-Finer (`H` cho private, `B` cho private_mixed),
6. ghi vào persistent submission folder trên Drive.

Không có GT private nên **không tính metric local**. Điểm chính thức phải lấy từ MVTec evaluation server.


In [ ]:
SUBMISSION_ROOT = DRIVE_OUT / "submission_folder"

def copy_merged_private_to_submission(
    category,
    split,
    merged_root,
):
    dst = (
        SUBMISSION_ROOT
        / "anomaly_images"
        / category
        / split
    )

    if dst.exists():
        shutil.rmtree(dst)

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copytree(
        merged_root,
        dst,
    )

    return dst

def binarize_private_submission(
    category,
    split,
    continuous_dir,
):
    out = (
        SUBMISSION_ROOT
        / "anomaly_images_thresholded"
        / category
        / split
    )

    if out.exists():
        shutil.rmtree(out)

    out.mkdir(
        parents=True,
        exist_ok=True,
    )

    amap_paths = sorted(
        continuous_dir.glob("*.tiff")
    )

    assert amap_paths

    mebin = MEBin(
        [str(p) for p in amap_paths],
        class_name=category,
    )

    binary_maps, thresholds = mebin.binarize_anomaly_maps()

    for src, binary in zip(amap_paths, binary_maps):
        cv2.imwrite(
            str(out / f"{src.stem}.png"),
            binary,
        )

    pd.DataFrame({
        "image": [p.stem for p in amap_paths],
        "threshold_mebin": thresholds,
    }).to_csv(
        out / "thresholds.csv",
        index=False,
    )

    return out

def run_private_split(category, split):
    patch_root = infer_split_to_patch_tiffs(
        category,
        split,
        force=True,
    )

    merge_selected_category(
        category,
        split,
        patch_root,
    )

    continuous_submission_dir = copy_merged_private_to_submission(
        category,
        split,
        patch_root,
    )

    coarse_tmp = (
        CATEGORY_OUT
        / f"{split}_binary_coarse"
    )

    # Tạo coarse ở temp Drive dir để SAM đọc.
    if coarse_tmp.exists():
        shutil.rmtree(coarse_tmp)

    amap_paths = sorted(
        continuous_submission_dir.glob("*.tiff")
    )

    mebin = MEBin(
        [str(p) for p in amap_paths],
        class_name=category,
    )

    binary_maps, thresholds = mebin.binarize_anomaly_maps()

    coarse_tmp.mkdir(
        parents=True,
        exist_ok=True,
    )

    for src, binary in zip(amap_paths, binary_maps):
        cv2.imwrite(
            str(coarse_tmp / f"{src.stem}.png"),
            binary,
        )

    final_submission_dir = (
        SUBMISSION_ROOT
        / "anomaly_images_thresholded"
        / category
        / split
    )

    if RUN_SAM_FINER:
        samfiner_selected(
            category,
            split,
            coarse_tmp,
            final_submission_dir,
        )
    else:
        if final_submission_dir.exists():
            shutil.rmtree(final_submission_dir)
        shutil.copytree(
            coarse_tmp,
            final_submission_dir,
        )

    shutil.rmtree(coarse_tmp, ignore_errors=True)

    n_cont = len(list(continuous_submission_dir.glob("*.tiff")))
    n_bin = len(list(final_submission_dir.glob("*.png")))

    assert n_cont == n_bin > 0

    print(
        f"PRIVATE READY {category}/{split}: "
        f"{n_cont} continuous + {n_bin} binary"
    )

if RUN_PRIVATE_INFERENCE:
    for split in [
        "test_private",
        "test_private_mixed",
    ]:
        run_private_split(
            CATEGORY,
            split,
        )
else:
    print(
        "SKIP PRIVATE. "
        "Set RUN_PRIVATE_INFERENCE=True ở Cell 1 khi cần."
    )


## 14. Tổng hợp metric các category đã chạy

Cell không dùng GPU. Khi đủ 8 category, tạo:
- `robis_public_metrics.csv`
- dòng `macro_mean`

Các điểm này là **local test_public**, không phải private leaderboard.


In [ ]:
METRIC_DIR = DRIVE_OUT / "metrics"

rows = []
missing = []

for category in ALL_CATEGORIES:
    path = METRIC_DIR / f"{category}.csv"

    if path.is_file():
        one = pd.read_csv(path)
        assert len(one) == 1
        rows.append(one.iloc[0].to_dict())
    else:
        missing.append(category)

summary_df = pd.DataFrame(rows)

if not summary_df.empty:
    order = {
        c: i
        for i, c in enumerate(ALL_CATEGORIES)
    }

    summary_df["_order"] = summary_df["Category"].map(order)

    summary_df = (
        summary_df
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

display(summary_df)

print("Completed:", summary_df["Category"].tolist() if not summary_df.empty else [])
print("Missing  :", missing)

if len(missing) == 0:
    metric_cols = [
        c for c in [
            "AU_PRO_0.05",
            "AU_PRO_0.05_pct",
            "pixel_F1_max",
            "pixel_F1_max_pct",
            "SegF1",
            "SegF1_pct",
            "ClassF1",
            "ClassF1_pct",
        ]
        if c in summary_df.columns
    ]

    macro = {"Category": "macro_mean"}

    for col in metric_cols:
        macro[col] = float(summary_df[col].mean())

    final_df = pd.concat([
        summary_df,
        pd.DataFrame([macro]),
    ], ignore_index=True)

    final_path = DRIVE_OUT / "robis_public_metrics.csv"

    final_df.to_csv(
        final_path,
        index=False,
    )

    print("8/8 COMPLETE:", final_path)
    display(final_df)


## 15. Package submission khi đủ 8 category

Archive cuối cùng có dạng:

```text
submission.tar.gz
├── anomaly_images/
│   └── <category>/
│       ├── test_private/*.tiff
│       └── test_private_mixed/*.tiff
└── anomaly_images_thresholded/
    └── <category>/
        ├── test_private/*.png
        └── test_private_mixed/*.png
```


In [ ]:
def submission_complete():
    problems = []

    for category in ALL_CATEGORIES:
        for split in [
            "test_private",
            "test_private_mixed",
        ]:
            cont_dir = (
                SUBMISSION_ROOT
                / "anomaly_images"
                / category
                / split
            )

            bin_dir = (
                SUBMISSION_ROOT
                / "anomaly_images_thresholded"
                / category
                / split
            )

            n_cont = (
                len(list(cont_dir.glob("*.tiff")))
                if cont_dir.exists()
                else 0
            )

            n_bin = (
                len(list(bin_dir.glob("*.png")))
                if bin_dir.exists()
                else 0
            )

            if n_cont == 0 or n_bin == 0 or n_cont != n_bin:
                problems.append(
                    (category, split, n_cont, n_bin)
                )

    return problems

problems = submission_complete()

if problems:
    print("Submission chưa đủ 8 category:")
    for row in problems:
        print(row)

else:
    archive_path = DRIVE_OUT / "submission.tar.gz"

    if archive_path.exists():
        archive_path.unlink()

    with tarfile.open(
        archive_path,
        "w:gz",
    ) as tf:
        tf.add(
            SUBMISSION_ROOT / "anomaly_images",
            arcname="anomaly_images",
        )

        tf.add(
            SUBMISSION_ROOT / "anomaly_images_thresholded",
            arcname="anomaly_images_thresholded",
        )

    print("SUBMISSION READY:", archive_path)


## 16. Audit selected category

Mục tiêu: lưu lại bằng chứng reproducibility cho từng category:
- commit RoBiS,
- SHA256 model,
- số ảnh native,
- số crop train,
- số public continuous maps,
- số public binary masks,
- hyperparameter chính.


In [ ]:
final_ckpt = CHECKPOINT_ROOT / CATEGORY / "model.pth"

audit = {
    "category": CATEGORY,
    "robis_commit": ROBIS_COMMIT,
    "checkpoint_exists": final_ckpt.is_file(),
    "checkpoint_sha256": (
        sha256_file(final_ckpt)
        if final_ckpt.is_file()
        else None
    ),
    "native_counts": audit_native,
    "processed_counts": processed_counts,
    "public_continuous_maps": (
        len(list(PUBLIC_CONT_DIR.rglob("*.tiff")))
        if PUBLIC_CONT_DIR.exists()
        else 0
    ),
    "public_final_binary_maps": (
        len(list(PUBLIC_FINAL_DIR.rglob("*.png")))
        if PUBLIC_FINAL_DIR.exists()
        else 0
    ),
    "settings": {
        "encoder": "dinov2reg_vit_base_14",
        "input_size": INPUT_SIZE,
        "crop_size": CROP_SIZE,
        "inp_num": INP_NUM,
        "epochs": EPOCHS,
        "train_batch": TRAIN_BATCH,
        "seed": SEED,
        "swin_window": SWIN_WINDOW,
        "swin_overlap": SWIN_OVERLAP,
        "map_size": MAP_SIZE,
        "gaussian_kernel": 5,
        "gaussian_sigma": 4,
        "binarization": "MEBin OR mean+3std",
        "sam_public_private": "vit_h except rice",
        "sam_private_mixed": "vit_b except rice",
    },
}

audit_dir = DRIVE_OUT / "audit"
audit_dir.mkdir(parents=True, exist_ok=True)

audit_path = audit_dir / f"{CATEGORY}.json"

audit_path.write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8",
)

print(json.dumps(audit, indent=2))
print("AUDIT:", audit_path)


## Workflow 8 category

Không train batch 8 model trong một runtime.

```text
CATEGORY = "can"
Restart runtime → Run all
    ↓
CATEGORY = "fabric"
Restart runtime → Run all
    ↓
...
CATEGORY = "walnuts"
Restart runtime → Run all
```

Khi mục tiêu trước mắt chỉ là **đánh giá khoa học trên test_public**, giữ:

```python
RUN_PUBLIC_EVAL = True
RUN_MEBIN = True
RUN_SAM_FINER = True
RUN_PRIVATE_INFERENCE = False
```

Khi cần tạo submission cho MVTec server, bật:

```python
RUN_PRIVATE_INFERENCE = True
```

### Mốc đối chiếu từ repo/paper RoBiS

Private leaderboard mà tác giả báo cáo:

| Split | mean AU-PRO@0.05 | mean SegF1 |
|---|---:|---:|
| `test_private` | 67.25 | 51.00 |
| `test_private_mixed` | 59.65 | 46.52 |

**Không kỳ vọng local `test_public` phải bằng đúng các số trên**, vì split khác nhau.
